### Creating Reference Delta Table To Be Used In This Shallow & Deep Clone Lab

- **invoices_tbl_clone_reference** will be used as a reference table for this lab
- For that first we are creating invoices_tbl_clone_reference and performing some operations over it to have some versions/history.

In [0]:
-- This is not a clone table, it will be used to create clone tables in this lab.

CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_tbl_clone_reference AS
SELECT
  *
FROM
  PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`;

In [0]:
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_tbl_clone_reference
LIMIT 5;

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_clone_reference;

In [0]:
-- Delete
DELETE FROM
  delta_catalog.delta_db.invoices_tbl_clone_reference
WHERE
  customer_id = 4;

-- Update
UPDATE
  delta_catalog.delta_db.invoices_tbl_clone_reference
SET
  quantity = 1000
WHERE
  customer_id = 5;

-- Insert
INSERT INTO delta_catalog.delta_db.invoices_tbl_clone_reference
VALUES (101, '1000', 'Female', 20, 'Category 1', 1000, 1000, 'Credit Card', '2022-01-01', 'Mall 1', null);

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_clone_reference;
-- DROP TABLE delta_catalog.delta_db.invoices_shallow_clone_v0_tbl;

### Shallow Clone

A shallow clone in Delta Lake creates a new table that references the original table's data files without copying them. This allows for fast table creation and minimal storage usage. The clone maintains its own history and operations, so changes to the source table after cloning do not affect the clone, and vice versa. Shallow clones are useful for testing, experimentation, and creating temporary tables without duplicating large datasets.

In [0]:
CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_tbl_shallow_clone SHALLOW CLONE delta_catalog.delta_db.invoices_tbl_clone_reference;

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_shallow_clone;

-- operationParameters column in history gives imp metadata.

#### Updating customer_id = 10 from reference table to verify that the changes occurs in shallow clone table as well or not.

In [0]:
UPDATE
  delta_catalog.delta_db.invoices_tbl_clone_reference
SET
  quantity = 2000
WHERE
  customer_id = 10;

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_clone_reference;

-- So update is reflected in the history of reference table.

In [0]:
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_tbl_clone_reference
WHERE
  customer_id = 10;

-- Record is updated in Reference table.

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_shallow_clone;

-- So, Update customer_id = 10 is not there in shallow clone table history.
-- Because changes made to reference table does'nt effect shallow clone table eventhough it refereces it.
-- Once shallow clone table is created, it will maintain its own history.

In [0]:
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_tbl_shallow_clone
WHERE
  customer_id = 10;

-- So, changes made to reference table has'nt effected shallow clone table.

**Conclusion On Shallow Clone**

- **Independence:** Even though a shallow clone(here `invoices_tbl_shallow_clone`) references the source table's(here `invoices_tbl_clone_reference`) underlying data files, But after cloning it maintains its own distinct history and set of operations.
- **No Cross-Effect:** Any subsequent modifications, updates, or deletions performed on the source table will not affect the shallow clone table and **vice versa**.
- **Encapsulation:** Once the clone is created, it functions as an independent entity with its own lifecycle, separate from the source table's ongoing activity.

**This is the state of invoices_tbl_shallow_clone data files in ADLS**
- As we havnt made any operation over it yet so there is no parquet files because it reference to source table parquet files.
- Now, after cloning whatever operation we perform on shallow clone table will create its own logs and data files.

![image_1782051776230.png](NB7_images/image_1782051776230.png "image_1782051776230.png")

#### Now, Deleting customer_id = 99 from shallow clone table to verify that the changes occurs in reference table `(Vice Versa)`.

In [0]:
DELETE FROM
  delta_catalog.delta_db.invoices_tbl_shallow_clone
WHERE
  customer_id = 99;

In [0]:
SELECT * FROM
  delta_catalog.delta_db.invoices_tbl_shallow_clone
WHERE
  customer_id = 99;

-- returns nothing, record is deleted from shallow clone table.

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_shallow_clone;

-- New version history is also there for delete operation.

In [0]:
SELECT * FROM
  delta_catalog.delta_db.invoices_tbl_clone_reference
WHERE
  customer_id = 99;

-- As expected, Record is present in reference table.

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_clone_reference;

-- As expected, there is no delete history created in reference table.

**This is the state of invoices_tbl_shallow_clone data files in ADLS (After Delete Operation)**
- There is only a deletion vector added and no parquet because delete operation doesnt generate any.
- If it would have been insert/update operation then there would be parquet file added. 

![image_1782051863219.png](NB7_images/image_1782051863219.png "image_1782051863219.png")

#### Shallow Clone From Older Version Of Source/Reference Table

In [0]:
CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_tbl_shallow_clone_v0 SHALLOW CLONE delta_catalog.delta_db.invoices_tbl_clone_reference VERSION AS OF 0;

-- alternatively
-- CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_tbl_shallow_clone_v0 SHALLOW CLONE delta_catalog.delta_db.invoices_tbl_clone_reference TIMESTAMP AS OF '2026-06-20T16:48:09.000+00:00';

In [0]:
-- getting the data from latest version of invoices_tbl_shallow_clone_v0 table which is cloned from 0th version of reference table(which wont have any operations).

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_tbl_shallow_clone_v0
WHERE 
    customer_id in (4, 5, 10, 101);

In [0]:
-- getting the data from latest version of reference table for customer_id's we made changes on.

SELECT
  *
FROM
  delta_catalog.delta_db.invoices_tbl_clone_reference -- (latest version)
WHERE 
    customer_id in (4, 5, 10, 101);

-- The Records will not match with invoices_tbl_shallow_clone_v0(i.e 0th version of reference tbl) as we had multiple changes on reference table 

#### Time Travel On Shallow Clone

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_clone_reference;

In [0]:
CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_tbl_shallow_clone_tt SHALLOW CLONE
delta_catalog.delta_db.invoices_tbl_clone_reference;

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_shallow_clone_tt;

**Note :-**
- As seen above after creating shallow clone it create its own history and doesnt inherit source/reference table history.

#### Vacuum Behaviour On Source/Reference Table

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_clone_reference;

**This are the list of operation we performed on source/reference(invoices_tbl_clone_reference) table so far**

| version	| peration                          |
| --- | --- |
| 4	        | UPDATE                            |
| 3	        | WRITE                             |
| 2	        | UPDATE                            |
| 1	        | DELETE                            |
| 0	        | CREATE OR REPLACE TABLE AS SELECT |

**And this is the ADLS Location of invoices_tbl_clone_reference table containing multiple data files as per above operations**

![image_1782110315931.png](NB7_images/image_1782110315931.png "image_1782110315931.png")

- Now in v3 insert/write operation we appended just one row with customer_id = 101, which added one parquet file.
- To demonstrate Vacuum behaviour on source/reference table will delete this record to see if that file gets deleted or not.

In [0]:
-- Delete
DELETE FROM
  delta_catalog.delta_db.invoices_tbl_clone_reference
WHERE
  customer_id = 101;

**After delete just one deletion_vector is added(last one) as usual**

![image_1782110566477.png](NB7_images/image_1782110566477.png "image_1782110566477.png")

In [0]:
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_tbl_clone_reference
WHERE
  customer_id = 101;

-- So record with customer_id = 101 is deleted from invoices_tbl_clone_reference.

In [0]:
-- This Code was runing(with serverless) but Vacuum was'nt deleting the unused files from the adls as expected.

-- STEP 1: Set the physical retention duration property natively on the table level (Serverless option)
ALTER TABLE delta_catalog.delta_db.invoices_tbl_clone_reference 
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

-- STEP 2: Run VACUUM without the "RETAIN" clause 
-- (It will automatically read the 0 hours property we just set above)
VACUUM delta_catalog.delta_db.invoices_tbl_clone_reference;

-- So, used global setting with user managed cluster

SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM delta_catalog.delta_db.invoices_tbl_clone_reference RETAIN 0 HOURS;

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_clone_reference;

**After Vacuum the parquet file(part-00000-06a04adf) is still not deleted**

- Although some deletion vector got cleaned as seen on previous image its 4 now its 2.
- The reason parquet file is still there, because we have used this `invoices_tbl_clone_reference` delta table to shallow clone couple of tables. 
- This shallow clones still consumes that deleted record so, thats why it needs that parquet file.
- Workaround is to delete that record from all shallow clone tables created from source `invoices_tbl_clone_reference`.

![image_1782110907470.png](NB7_images/image_1782110907470.png "image_1782110907470.png")

In [0]:
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_tbl_shallow_clone
WHERE
  customer_id = 101;

-- So record with customer_id = 101 exist here.

In [0]:
-- invoices_tbl_shallow_clone
DELETE FROM
  delta_catalog.delta_db.invoices_tbl_shallow_clone
WHERE
  customer_id = 101;

-- invoices_tbl_shallow_clone_tt
DELETE FROM
  delta_catalog.delta_db.invoices_tbl_shallow_clone_tt
WHERE
  customer_id = 101;

-- invoices_tbl_shallow_clone_v0
DELETE FROM
  delta_catalog.delta_db.invoices_tbl_shallow_clone_v0
WHERE
  customer_id = 101;

In [0]:
-- This Code was runing(with serverless) but Vacuum was'nt deleting the unused files from the adls as expected.

-- STEP 1: Set the physical retention duration property natively on the table level (Serverless option)
ALTER TABLE delta_catalog.delta_db.invoices_tbl_clone_reference 
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

-- STEP 2: Run VACUUM without the "RETAIN" clause 
-- (It will automatically read the 0 hours property we just set above)
VACUUM delta_catalog.delta_db.invoices_tbl_clone_reference;

-- So, used global setting with user managed cluster

SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM delta_catalog.delta_db.invoices_tbl_clone_reference RETAIN 0 HOURS;

**Finally, After deleting all occurrence of 101 record from all shallow table refrencing the source table `invoices_tbl_clone_reference` and running vacuum on the source table, we can see that the parquet file(part-00000-06a04adf) is deleted from the adls path.**

![image_1782112847730.png](NB7_images/image_1782112847730.png "image_1782112847730.png")

### Deep Clone

Deep Clone in Delta Lake creates a full, independent copy of a table, including all data files and metadata. Unlike shallow clone, which references source data files, deep clone copies everything to a new location. After cloning, the deep clone table maintains its own history and operations, and changes to either the source or clone do not affect the other. Deep Clone is useful for backup, disaster recovery, and creating isolated test environments.

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_clone_reference;

In [0]:
CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_tbl_deep_clone DEEP CLONE
delta_catalog.delta_db.invoices_tbl_clone_reference;

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_deep_clone;

In [0]:
-- Show all tables that have been dropped in the database, but can be undroped
SHOW TABLES DROPPED IN delta_catalog.delta_db;

-- There is no supported bulk SQL command to purge all dropped tables from a schema.
-- Dropped managed tables age out automatically after the retention period.

**This is the state of invoices_tbl_deep_clone data files in ADLS**

- Deep Clone create exact copy of data/parquet files from the source(`invoices_tbl_clone_reference`) table parquet files.
- Now, after cloning whatever operation we perform on deep clone table will create its own logs and data files.

**invoices_tbl_deep_clone**
![image_1782116502032.png](NB7_images/image_1782116502032.png "image_1782116502032.png")

**invoices_tbl_clone_reference**
![image_1782112847730.png](NB7_images/image_1782112847730.png "image_1782112847730.png")

#### Updating customer_id = 6 In deep clone table to verify that the changes occurs in source/reference table as well or not.

In [0]:
UPDATE
  delta_catalog.delta_db.invoices_tbl_deep_clone
SET
  quantity = 2000
WHERE
  customer_id = 6;

In [0]:
SELECT
  *
from
  delta_catalog.delta_db.invoices_tbl_deep_clone
WHERE customer_id = 6;

-- So, recors is updated in deep clone

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_deep_clone;

In [0]:
SELECT
  *
from
  delta_catalog.delta_db.invoices_tbl_clone_reference
WHERE customer_id = 6;

-- As expected, update/or any operation on deep clone does not affect the source/reference table
-- and vice versa.

In [0]:
DESCRIBE HISTORY delta_catalog.delta_db.invoices_tbl_clone_reference;

**Conclusion On Deep Clone**

- **Independence:** Deep clone copies all metadata ad data files from source to its location and after cloning it maintains its own distinct history and set of operations.
- **No Cross-Effect:** Any subsequent modifications, updates, or deletions performed on the source table will not affect the deep clone table and **vice versa**.
- **Encapsulation:** Once the clone is created, it functions as an independent entity with its own lifecycle, separate from the source table's ongoing activity.

**Note :-**
- Vacuum explaination is not required for deep clone becuase it copies all all data file from source, unlike shallow clone which references it from source.
- Also to remind, we performed vacuum on reference table and shallow clone was referencing those data files while in deep clone it copies all file so that scenario is not applicable here. 

### CTAS(CREATE TABLE AS SELECT) vs. Deep Clone

* While both result in tables that look identical in output, they differ significantly in capability and process:
* **Handling of Properties**: **CTAS** creates a new table based on the output of a query, causing it to lose metadata, partitioning properties, and constraints from the source table. **Deep Clone** is more robust as it automatically clones the metadata, data, and existing table properties.
* **Incremental Processing**: A key advantage of **Deep Clone** is its support for **incremental syncing** . In scenarios like disaster recovery, if you need to update a replica, Deep Clone only copies the incremental changes (updates or deletes) rather than re-copying the entire dataset, making it far more performant for maintaining replicas.
* **Incremental Processing Limitation**: INSERT / UPDATE / MERGE / DELETE operations are synced incrementally. However, SCHEMA changes or changes in PARTITIONING, COLUMN changes will trigger a full DEEP CLONE